# Perhitungan Sentralitas

## Import Library

In [7]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

In [12]:
# Pengaturan tampilan
pd.set_option('display.max_colwidth', None)

# Float 8 desimal
pd.options.display.float_format = '{:.8f}'.format

# Membaca CSV
df = pd.read_csv('../data/processed/edges.csv')

# Konversi numerik
df['Weight'] = pd.to_numeric(df['Weight'])
df['Inverse_Weight'] = pd.to_numeric(df['Inverse_Weight'])

# Pembulatan
df['Inverse_Weight'] = df['Inverse_Weight'].round(8)

# Urutkan berdasarkan Weight terbesar
df = df.sort_values(
    by='Weight',
    ascending=False
).reset_index(drop=True)

# Output
display(df)

,Guru,Murid,Weight,Inverse_Weight
0,ابن عمر,نافع,11346,0.00008814
1,معمر,عبد الرزاق,6402,0.00015620
2,ابو هشام بن عروة,هشام بن عروة,6390,0.00015649
3,ابن عباس,عكرمة,5456,0.00018328
4,ابن عباس,سعيد بن جبير,4663,0.00021445
...,...,...,...,...
776944,المستلم بن سعيد,حماد بن سلمة,1,1.00000000
776945,عبد العزيز,المستلم بن سعيد,1,1.00000000
776946,الحجاج,عبد العزيز,1,1.00000000
776947,انس بن مالك,الحجاج,1,1.00000000


## Load Data

In [16]:
print(df.dtypes)

Guru                  str
Murid                 str
Weight              int64
Inverse_Weight    float64
dtype: object


In [17]:
df['Weight'] = pd.to_numeric(df['Weight'])
df['Inverse_Weight'] = pd.to_numeric(df['Inverse_Weight'])
print(df.dtypes)

Guru                  str
Murid                 str
Weight              int64
Inverse_Weight    float64
dtype: object


### Bentuk graph

In [18]:
G = nx.from_pandas_edgelist(
    df,
    source='Murid',
    target='Guru',
    edge_attr=['Weight', 'Inverse_Weight'],
    create_using=nx.DiGraph()
)

In [19]:
print("Jumlah node :", G.number_of_nodes())
print("Jumlah edge :", G.number_of_edges())

Jumlah node : 177212
Jumlah edge : 776949


In [20]:
display(df)

,Guru,Murid,Weight,Inverse_Weight
0,ابن عمر,نافع,11346,0.00008814
1,معمر,عبد الرزاق,6402,0.00015620
2,ابو هشام بن عروة,هشام بن عروة,6390,0.00015649
3,ابن عباس,عكرمة,5456,0.00018328
4,ابن عباس,سعيد بن جبير,4663,0.00021445
...,...,...,...,...
776944,المستلم بن سعيد,حماد بن سلمة,1,1.00000000
776945,عبد العزيز,المستلم بن سعيد,1,1.00000000
776946,الحجاج,عبد العزيز,1,1.00000000
776947,انس بن مالك,الحجاج,1,1.00000000


In [21]:
# Hapus self-loop
G.remove_edges_from(nx.selfloop_edges(G))

In [22]:
print("Jumlah node setelah hapus self-loop:", G.number_of_nodes())
print("Jumlah edge setelah hapus self-loop:", G.number_of_edges())

Jumlah node setelah hapus self-loop: 177212
Jumlah edge setelah hapus self-loop: 776949


## Menghitung Sentralitas

### Menghitung In Degree, Out Degree, Degree, dan Degree Centrality

In [23]:
# Degree (node atau total hubungan)
degree_dict = dict(G.degree())


# In-Degree (peran perawi sebagai Guru)
# In-Degree = jumlah murid
in_degree_dict = dict(G.in_degree())


# Out-Degree (peran perawi sebagai Murid)
# Out-Degree = jumlah guru
out_degree_dict = dict(G.out_degree())


# Degree Centrality (tingkat keterhubungan node dalam jaringan.)
degree_centrality_dict = nx.degree_centrality(G)

# Gabungkan ke DataFrame

centrality_df = pd.DataFrame({
    'Perawi': list(G.nodes()),

    'In_Degree_Jumlah_Murid': [
        in_degree_dict[node]
        for node in G.nodes()
    ],

    'Out_Degree_Jumlah_Guru': [
        out_degree_dict[node]
        for node in G.nodes()
    ],

    'Degree_Total_Hubungan': [
        degree_dict[node]
        for node in G.nodes()
    ],

    'Degree_Centrality': [
        degree_centrality_dict[node]
        for node in G.nodes()
    ]
})

# Urutkan berdasarkan Degree Centrality
centrality_df = centrality_df.sort_values(
    by='Degree_Centrality',
    ascending=False
)


# Reset index
centrality_df = centrality_df.reset_index(drop=True)


# Tampilkan hasil
display(centrality_df.head(20))

,Perawi,In_Degree_Jumlah_Murid,Out_Degree_Jumlah_Guru,Degree_Total_Hubungan,Degree_Centrality
0,ابو هريرة,3530,1291,4821,0.02720486
1,سفيان,2294,2391,4685,0.02643741
2,شعبة,2123,2198,4321,0.02438336
3,ابن عباس,2662,947,3609,0.02036555
4,عائشة,2334,733,3067,0.01730705
5,الاعمش,1748,1306,3054,0.01723369
6,ابن عمر,2246,796,3042,0.01716598
7,ابو عبد الله الحافظ,786,2247,3033,0.01711519
8,الزهري,1554,1467,3021,0.01704747
9,عبد الله,1491,1364,2855,0.01611074


### Menghitung Eigenvector Centrality (Kualitas koneksi)

In [24]:
# === Eigenvector Centrality (Power Iteration)
try:
    eigenvector_centrality = nx.eigenvector_centrality(
        G,
        max_iter=5000,
        tol=1e-05,
        weight='Weight'
    )
    print("Eigenvector Centrality selesai.")
except nx.PowerIterationFailedConvergence as e:
    eigenvector_centrality = {}
    print(f"Proses Eigenvector Centrality gagal konvergen: {e}")

# Tetap memakai nama lama agar sel berikutnya masih kompatibel
eigenvector_dict = eigenvector_centrality

# Ubah hasil Power Iteration menjadi DataFrame
eigenvector_df = pd.DataFrame({
    'Perawi': list(eigenvector_centrality.keys()),
    'Eigenvector_Centrality': list(eigenvector_centrality.values())
})

if not eigenvector_df.empty:
    eigenvector_df = eigenvector_df.sort_values(
        by='Eigenvector_Centrality',
        ascending=False
    ).reset_index(drop=True)

display(eigenvector_df.head(20))


# # === Eigenvector Centrality (NumPy version)
# try:
#     eigenvector_numpy_centrality = nx.eigenvector_centrality_numpy(
#         G,
#         weight='Weight'
#     )
#     print("✅ Eigenvector Numpy Centrality selesai.")
# except Exception as e:
#     eigenvector_numpy_centrality = {}
#     print(f"Proses Eigenvector Numpy Centrality gagal: {e}")

# # Ubah hasil NumPy menjadi DataFrame
# eigenvector_numpy_df = pd.DataFrame({
#     'Perawi': list(eigenvector_numpy_centrality.keys()),
#     'Eigenvector_Numpy_Centrality': list(eigenvector_numpy_centrality.values())
# })

# if not eigenvector_numpy_df.empty:
#     eigenvector_numpy_df = eigenvector_numpy_df.sort_values(
#         by='Eigenvector_Numpy_Centrality',
#         ascending=False
#     ).reset_index(drop=True)

# display(eigenvector_numpy_df.head(20))

Eigenvector Centrality selesai.


,Perawi,Eigenvector_Centrality
0,ابو هريرة,0.51953442
1,عائشة,0.48361870
2,ابن عمر,0.38208942
3,ابن عباس,0.26586841
4,انس,0.17225403
5,عمر,0.15470174
6,انس بن مالك,0.12854060
7,سعيد بن المسيب,0.12310829
8,الزهري,0.11976709
9,عروة,0.11490185


In [25]:
# mengecek jumlah node setelah semua proses
print(G.number_of_nodes())

177212


In [26]:
# mengecek data  yang digunakan untuk menghitung eigenvector centrality
print(len(eigenvector_dict))

177212


### Menghitung data dengan data teratas untuk Closeness dan Between 

In [22]:
# # Ambil 40000 data teratas
# df_40000 = df.head(40000)

# # Buat graph dari kolom Guru dan Murid
# G_sample = nx.from_pandas_edgelist(
#     df_40000,
#     source='Guru',
#     target='Murid',
#     edge_attr=['Weight', 'Inverse_Weight'],
#     create_using=nx.DiGraph()
# )

# # Tampilkan jumlah node dan edge
# print("Jumlah Node :", G_sample.number_of_nodes())
# print("Jumlah Edge :", G_sample.number_of_edges())

In [23]:
# display(df_40000)

### Menghitung Closeness Centrality

In [ ]:
import networkx as nx
import pandas as pd
from tqdm import tqdm

# ======================================
# Ambil 3000 data teratas
# ======================================

df_sample = df.head(20000).copy()
display(df_sample)

# ======================================
# Pastikan Inverse_Weight numerik
# ======================================

df_sample['Inverse_Weight'] = pd.to_numeric(
    df_sample['Inverse_Weight']
)

# ======================================
# Hindari nilai terlalu kecil
# ======================================

df_sample['Inverse_Weight'] = (
    df_sample['Inverse_Weight']
    .clip(lower=0.01)
)

# ======================================
# Build graph Guru -> Murid
# ======================================

G_sample = nx.from_pandas_edgelist(
    df_sample,
    source='Guru',
    target='Murid',
    edge_attr=['Weight', 'Inverse_Weight'],
    create_using=nx.DiGraph()
)

print("Jumlah Node :", G_sample.number_of_nodes())
print("Jumlah Edge :", G_sample.number_of_edges())

# ======================================
# Hitung Closeness Centrality
# ======================================

raw_closeness = {}

for node in tqdm(
    G_sample.nodes(),
    desc="Menghitung Closeness Centrality"
):

    raw_closeness[node] = nx.closeness_centrality(
        G_sample,
        u=node,
        distance='Inverse_Weight'
    )

print("✅ Closeness Centrality selesai.")

# ======================================
# Normalisasi Min-Max (0 - 1)
# ======================================

min_val = min(raw_closeness.values())
max_val = max(raw_closeness.values())

normalized_closeness = {

    node: (
        (val - min_val) / (max_val - min_val)
        if max_val > min_val else 1.0
    )

    for node, val in raw_closeness.items()
}

print("✅ Normalisasi Closeness selesai.")

# ======================================
# Ubah hasil menjadi DataFrame
# ======================================

closeness_df = pd.DataFrame({
    'Perawi': list(normalized_closeness.keys()),
    'Closeness_Centrality': list(normalized_closeness.values())
})

# ======================================
# Urutkan dari terbesar
# ======================================

closeness_df = closeness_df.sort_values(
    by='Closeness_Centrality',
    ascending=False
).reset_index(drop=True)

# ======================================
# Tampilkan 20 teratas
# ======================================

display(closeness_df.head(20))

### Menghitung Between

In [25]:
import networkx as nx
import pandas as pd
from tqdm import tqdm
from datetime import datetime


# Ambil 20000 data teratas
df_sample = df.head(20000).copy()

# Pastikan Inverse_Weight numerik
df_sample['Inverse_Weight'] = pd.to_numeric(
    df_sample['Inverse_Weight']
)


# Hindari nilai terlalu kecil
df_sample['Inverse_Weight'] = (
    df_sample['Inverse_Weight']
    .clip(lower=0.01)
)


# Build graph Guru -> Murid
G = nx.from_pandas_edgelist(
    df_sample,
    source='Guru',
    target='Murid',
    edge_attr=['Weight', 'Inverse_Weight'],
    create_using=nx.DiGraph()
)

print("Graph berhasil dibuat")
print("Jumlah Node :", G.number_of_nodes())
print("Jumlah Edge :", G.number_of_edges())


# Timestamp mulai
start_time = datetime.now()

print(f"Betweenness Centrality started at: {start_time}")


# Fungsi Betweenness Centrality
# dengan progress bar
def betweenness_centrality_with_progress(
    G,
    normalized=True,
    weight=None
):

    nodes = list(G.nodes())

    bet = dict.fromkeys(nodes, 0.0)

    for s in tqdm(
        nodes,
        desc="Betweenness Centrality (Nodes)"
    ):

        # Hitung kontribusi betweenness
        contrib = nx.betweenness_centrality_subset(
            G,
            sources=[s],
            targets=nodes,
            normalized=normalized,
            weight=weight
        )

        # Gabungkan kontribusi
        for n, v in contrib.items():

            bet[n] += v

    return bet


# Hitung Betweenness Centrality
bet_node = betweenness_centrality_with_progress(
    G,
    normalized=True,
    weight='Inverse_Weight'
)

print("✅ Betweenness Centrality selesai.")


# Timestamp selesai
end_time = datetime.now()

print(f"Finished at: {end_time}")
print(f"Duration: {end_time - start_time}")


# Ubah hasil menjadi DataFrame
betweenness_df = pd.DataFrame({
    'Perawi': list(bet_node.keys()),
    'Betweenness_Centrality': list(bet_node.values())
})


# Urutkan dari terbesar
betweenness_df = betweenness_df.sort_values(
    by='Betweenness_Centrality',
    ascending=False
).reset_index(drop=True)

# Tampilkan 20 teratas
display(betweenness_df.head(20))

Graph berhasil dibuat
Jumlah Node : 7215
Jumlah Edge : 20000
Betweenness Centrality started at: 2026-05-19 09:41:50.910292


Betweenness Centrality (Nodes): 100%|██████████| 7215/7215 [04:03<00:00, 29.65it/s]

✅ Betweenness Centrality selesai.
Finished at: 2026-05-19 09:45:54.286423
Duration: 0:04:03.376131


,Perawi,Betweenness_Centrality
0,سفيان,0.448654
1,عبد الله,0.178975
2,ابو حنيفة,0.138927
3,ابو هريرة,0.128559
4,علي,0.118261
5,يحيي بن سعيد,0.108273
6,ابن عباس,0.083025
7,محمد بن ابراهيم,0.082844
8,شعبة,0.081177
9,عمر,0.077763


### Pagerank

In [26]:
import networkx as nx
import pandas as pd
from tqdm import tqdm
from datetime import datetime


# Ambil data keseluruhan
df_sample = df

# Pastikan Weight numerik

df_sample['Weight'] = pd.to_numeric(
    df_sample['Weight']
)

# Build graph Guru -> Murid

G_sample = nx.from_pandas_edgelist(
    df_sample,
    source='Guru',
    target='Murid',
    edge_attr=['Weight', 'Inverse_Weight'],
    create_using=nx.DiGraph()
)

print("✅ Graph berhasil dibuat")
print("Jumlah Node :", G_sample.number_of_nodes())
print("Jumlah Edge :", G_sample.number_of_edges())


# Timestamp mulai
start_time = datetime.now()

print(f"🚀 PageRank started at: {start_time}")


# Hitung PageRank
# Mengukur tingkat pengaruh
# seorang perawi berdasarkan
# koneksi dengan perawi penting lain

pagerank_dict = nx.pagerank(
    G_sample,
    alpha=0.85,
    weight='Weight'
)

print("PageRank selesai.")


# Timestamp selesai
end_time = datetime.now()

print(f"Finished at: {end_time}")
print(f"Duration: {end_time - start_time}")

# Ubah hasil menjadi DataFrame
pagerank_df = pd.DataFrame({
    'Perawi': list(pagerank_dict.keys()),
    'PageRank': list(pagerank_dict.values())
})

# Urutkan dari terbesar
pagerank_df = pagerank_df.sort_values(
    by='PageRank',
    ascending=False
).reset_index(drop=True)

# Tampilkan 20 teratas
display(pagerank_df.head(20))

✅ Graph berhasil dibuat
Jumlah Node : 177212
Jumlah Edge : 776949
🚀 PageRank started at: 2026-05-19 09:46:01.260637
PageRank selesai.
Finished at: 2026-05-19 09:46:06.764228
Duration: 0:00:05.503591


,Perawi,PageRank
0,ابو عبد الله الحافظ,0.008318
1,سفيان,0.004616
2,شعبة,0.004508
3,سليمان بن احمد,0.003463
4,ابو بكر بن ابو شيبة,0.002688
5,ابو داود,0.002654
6,مالك,0.002561
7,ابو العباس محمد بن يعقوب,0.002396
8,الاعمش,0.002274
9,الزهري,0.002117
